# WHW08 starter notebook: matrices as transformations

A 2x2 matrix takes a point $(x, y)$ (written as a column) to a new point. If you apply it to every
corner of a shape, you see what it does to the whole shape: stretch it, shear it, rotate it, flip
it. We're going to do that to the unit square, with corners $(0,0)$, $(1,0)$, $(1,1)$, $(0,1)$, and
three matrices:

$$A = \begin{pmatrix} 2 & 0 \\ 0 & 1 \end{pmatrix}, \qquad
B = \begin{pmatrix} 1 & 1 \\ 0 & 1 \end{pmatrix}, \qquad
C = \begin{pmatrix} 0 & -1 \\ 1 & 0 \end{pmatrix},$$

and also the products $AB$ and $BA$.

**New Python in this notebook:**

- `np.array` for a **2x2 matrix**, and for a **set of points** stored as the columns of an array
  (row 0 holds the $x$ values, row 1 holds the $y$ values).
- **Matrix products with `@`**: `M @ points` applies $M$ to every point at once, and `M @ N` is the
  matrix product $MN$.
- **`np.linalg.det`** to compute a determinant.
- **Plotting a closed polygon** from the rows of an array (you have to repeat the first corner
  at the end, or the last side is missing).

## 0. Setup

Here's where we do all of our python setup. We'll grab matplotlib for plotting, sympy for symbolic manipulation, and numpy for numerical python calculations

In [ ]:
%matplotlib inline
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from sympy import *
init_printing()

## 1. Predict first (before you compute anything)

1. For each of $A$, $B$, and $C$: in words, what do you expect it to do to the unit square? (Hint:
   work out where it sends the corners $(1,0)$ and $(0,1)$.)
2. For each of $A$, $B$, and $C$: what happens to the square's **area**? (Bigger, smaller, the
   same? By what factor?)
3. $AB$ means "do $B$ first, then $A$" (because $AB\vec v = A(B\vec v)$). Do you expect $AB$ and
   $BA$ to do the same thing to the square?

Double-click the cell below, type your prediction, and press Shift-Enter.

*(your prediction here)*

## 2. Demo: matrices and points in NumPy

We'll practice on a *different* matrix and a *different* shape: the matrix

$$M = \begin{pmatrix} 1.5 & 0.5 \\ 0.5 & 1 \end{pmatrix}$$

and the triangle with corners $(0,0)$, $(2,0)$, $(0,1)$.

A matrix in NumPy is a list of rows: `np.array([[1.5, 0.5], [0.5, 1.0]])`. (Same idea as
`{{1.5, 0.5}, {0.5, 1}}` in Mathematica.)

For a shape, we store **each corner as a column**. So row 0 is all the $x$ values, and row 1 is all
the $y$ values. Read the triangle below *down the columns*: $(0,0)$, then $(2,0)$, then $(0,1)$.

In [ ]:
M = np.array([[1.5, 0.5],
              [0.5, 1.0]])

triangle = np.array([[0, 2, 0],
                     [0, 0, 1]])

print('M =')
print(M)
print('triangle =')
print(triangle)
print('x values (row 0):', triangle[0])
print('y values (row 1):', triangle[1])

**Matrix multiplication is `@`** (not `*`, which multiplies element by element and is *not*
the matrix product). `M @ np.array([2, 0])` is $M$ times the column $(2, 0)$. And `M @ triangle`
multiplies $M$ by every column at once, so you get the image of every corner in one line.
Check that the second column of `M @ triangle` matches `M @ np.array([2, 0])`.

In [ ]:
print('M @ (2, 0) =', M @ np.array([2, 0]))
print('M @ triangle =')
print(M @ triangle)

**Plotting a shape.** `plt.plot(x_values, y_values)` connects the dots in order. If we hand it
the triangle's rows directly, it goes $(0,0) \to (2,0) \to (0,1)$ and stops: the last side is
missing.

In [ ]:
plt.plot(triangle[0], triangle[1], 'o-')
plt.axis('equal')
plt.title('Oops: not closed')
plt.show()

To close it, we repeat the first corner at the end. Here's a short function that does that, line
by line:

- `points[0]` is row 0 (all the $x$ values); `points[0, 0]` is the very first $x$ value.
- `np.append(points[0], points[0, 0])` makes a new array: all the $x$ values, then the first one
  again. Same for $y$.
- `style` is a matplotlib line style, like in WHW04: `'o-'` means "dots at the corners, joined
  by lines", and `'k--'` means "black dashed line".
- `plt.axis('equal')` (in the cell after) makes one unit on the $x$ axis the same length as one
  unit on the $y$ axis. Without it, a square might not *look* square, and you can't judge
  areas by eye.

You can use `plot_shape` for the rest of the notebook.

In [ ]:
def plot_shape(points, label, style):
    x_values = np.append(points[0], points[0, 0])
    y_values = np.append(points[1], points[1, 0])
    plt.plot(x_values, y_values, style, label=label)

plot_shape(triangle, 'triangle', 'o-')
plot_shape(M @ triangle, 'M @ triangle', 'o-')
plt.axis('equal')
plt.grid(True)
plt.legend()
plt.show()

**Determinants.** `np.linalg.det(M)` computes $\det M$. Computers do this with decimals, so you
might see something like `1.2500000000000002` instead of `1.25`; that last digit is round-off,
not a real difference.

Here's a fact we'll prove in class (section 6.7): **for a 2x2 matrix $M$, every shape's area gets
multiplied by $|\det M|$.** You don't need that class to use it here; just check it numerically.
The function `polygon_area` below measures the area of a shape stored the way we've been storing
them. You don't need to understand how it works (it's called the "shoelace formula"); treat it
as a ruler.

In [ ]:
def polygon_area(points):
    x_values = points[0]
    y_values = points[1]
    return 0.5 * np.abs(np.sum(x_values * np.roll(y_values, -1) - np.roll(x_values, -1) * y_values))

print('det(M)                =', np.linalg.det(M))
print('area of triangle      =', polygon_area(triangle))
print('area of M @ triangle  =', polygon_area(M @ triangle))
print('ratio                 =', polygon_area(M @ triangle) / polygon_area(triangle))

**Matrix times matrix.** `M @ N` is the product $MN$. Since $(MN)\vec v = M(N\vec v)$, applying
`M @ N` to the triangle is the same as applying $N$ first and then $M$. Here
$N = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}$, which swaps $x$ and $y$ (a mirror flip across
the line $y = x$). Notice its determinant is $-1$: the area doesn't change ($|{-1}| = 1$), and the
minus sign is the determinant's way of saying the shape got flipped over.

In [ ]:
N = np.array([[0, 1],
              [1, 0]])

print('(M @ N) @ triangle =')
print((M @ N) @ triangle)
print('M @ (N @ triangle) =')
print(M @ (N @ triangle))
print('det(N) =', np.linalg.det(N))

## 3. Your turn: the unit square

In the cell below, change the marked lines:

1. `square` should hold the corners of the unit square, $(0,0)$, $(1,0)$, $(1,1)$, $(0,1)$, as
   columns, in that order (going around the square, so the plot connects them correctly).
2. `A`, `B`, and `C` should be the matrices at the top of this notebook.
3. `AB` and `BA` should be the matrix products $AB$ and $BA$.

As handed to you, the cell runs, but `square` is the demo's triangle and every matrix is the
identity (which does nothing). That's on purpose: change it!

In [ ]:
# Change the line below.
square = np.array([[0, 2, 0],
                   [0, 0, 1]])

# Change the line below.
A = np.array([[1, 0],
              [0, 1]])
# Change the line below.
B = np.array([[1, 0],
              [0, 1]])
# Change the line below.
C = np.array([[1, 0],
              [0, 1]])

# Change the line below.
AB = A
# Change the line below.
BA = A

## 4. Now let's plot it and see if we were right.

The first plot shows the square and its images under $A$, $B$, and $C$. The second shows the
square and its images under $AB$ and $BA$. (Six shapes on one set of axes is too many to read.)
The original square is drawn last, as a black dashed line, so you can still see it where the
other shapes sit on top of its edges.

In [ ]:
plot_shape(A @ square, 'A', 'o-')
plot_shape(B @ square, 'B', 'o-')
plot_shape(C @ square, 'C', 'o-')
plot_shape(square, 'original square', 'k--')
plt.axis('equal')
plt.grid(True)
# Put the legend outside the plot, so it doesn't cover any corners.
plt.legend(loc='upper left', bbox_to_anchor=(1.02, 1))
plt.show()

plot_shape(AB @ square, 'AB', 'o-')
plot_shape(BA @ square, 'BA', 'o-')
plot_shape(square, 'original square', 'k--')
plt.axis('equal')
plt.grid(True)
# Put the legend outside the plot, so it doesn't cover any corners.
plt.legend(loc='upper left', bbox_to_anchor=(1.02, 1))
plt.show()

## 5. Determinants vs. areas

Compute $\det A$, $\det B$, $\det C$, $\det(AB)$, and $\det(BA)$ with `np.linalg.det`. The first
one is done for you; add the other four.

In [ ]:
print('det(A)  =', np.linalg.det(A))
# Add lines for det(B), det(C), det(AB), and det(BA).

And here are the areas, measured with the `polygon_area` ruler from the demo, so you can compare
them with the determinants.

In [ ]:
print('area of square        =', polygon_area(square))
print('area of A @ square    =', polygon_area(A @ square))
print('area of B @ square    =', polygon_area(B @ square))
print('area of C @ square    =', polygon_area(C @ square))
print('area of AB @ square   =', polygon_area(AB @ square))
print('area of BA @ square   =', polygon_area(BA @ square))

## 6. What you found

**(a)** Which of your predictions was wrong? (About the shapes, the areas, or $AB$ vs. $BA$.)

*(your answer here)*

**(b)** $AB$ and $BA$ give visibly different pictures but the same determinant. In two or three
sentences: what does the determinant actually tell you about a transformation, and what does it
*not* tell you?

*(your answer here)*

**(c)** One thing you tried that did not work (forgetting to repeat the first corner so the
polygon closes is the usual stumble; corners in the wrong order, `*` instead of `@`, and a square
that doesn't look square are others).

*(your answer here)*